# Lesson 7: Regression for Causal Adjustment

## Opening Story: Education and Earnings

Does education cause higher earnings? The answer seems obvious—college graduates earn more than high school graduates. But correlation isn't causation. People who attend college might be smarter, more motivated, or come from wealthier families. These factors affect both education and earnings.

The question is: can regression analysis help us estimate the causal effect of education on earnings? The answer is: yes, under certain conditions—but those conditions are often violated in practice.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Use regression as a tool for causal adjustment
2. Understand conditional expectations and the law of iterated expectations
3. Implement multiple regression for causal inference
4. Recognize when regression fails
5. Interpret regression coefficients causally under appropriate assumptions

---

## 7.1 Regression as Causal Adjustment

### The Core Idea

Under the **unconfoundedness assumption** (all confounders are observed and controlled), regression can estimate causal effects:

$$Y = \alpha + \tau T + \beta X + \epsilon$$

If this model is correctly specified:
- $\tau$ is the average treatment effect
- $\beta$ captures the effect of confounders
- $\epsilon$ is independent of $T$ and $X$

### When This Works

1. All confounders are measured and included in $X$
2. The functional form is correct (linear, additive)
3. No measurement error in confounders
4. No collinearity between $T$ and $X$

---

## 7.2 Conditional Expectations

### The Law of Iterated Expectations

$$E[Y | X] = E[E[Y | T, X] | X]$$

This means: the expected outcome given covariates is the average of the conditional treatment effects, weighted by the probability of treatment.

### The Regression Interpretation

In a correctly specified regression:
$$E[Y | T, X] = \alpha + \tau T + \beta X$$

The coefficient $\tau$ is the average treatment effect, conditional on $X$.

---

## 7.3 Multiple Regression

### The Workhorse

Multiple regression is the most common method for causal adjustment:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

np.random.seed(42)
n = 1000

# Generate data with confounding
ability = np.random.normal(0, 1, n)
education = 12 + 2 * ability + np.random.normal(0, 1, n)
earnings = 20000 + 5000 * education + 10000 * ability + np.random.normal(0, 5000, n)

# Omitted variable bias
reg_omit = sm.OLS(earnings, sm.add_constant(education)).fit()
print("Effect of education (omitting ability):")
print(f"  Coefficient: ${reg_omit.params[1]:.2f}")
print(f"  True effect: $5000")

# Complete regression
X_complete = np.column_stack([education, ability])
reg_complete = sm.OLS(earnings, sm.add_constant(X_complete)).fit()
print("\nEffect of education (controlling for ability):")
print(f"  Coefficient: ${reg_complete.params[1]:.2f}")

---

## 7.4 When Regression Fails

### 1. Omitted Variable Bias

If important confounders are missing, regression estimates are biased.

### 2. Functional Form Misspecification

If the true relationship is nonlinear but we fit a linear model, estimates can be biased.

### 3. Measurement Error

Errors in measuring confounders can cause attenuation bias (coefficients biased toward zero).

### 4. Perfect Collinearity

If treatment is perfectly predicted by covariates, we can't estimate its effect.

---

## 7.5 Case Study: Returns to Education

### The Question

What is the causal effect of an additional year of education on earnings?

### The Challenge

- Ability affects both education and earnings
- Family background affects both
- Motivation affects both

### The Solution

Use instrumental variables (covered in Lesson 9) or carefully control for observed confounders.

### Typical Findings

- Naive regression: ~10% return per year
- After controlling for ability: ~7-8% return
- IV estimates: ~5-15% return (varies by study)

---

## 7.6 Python Workshop: Regression Diagnostics

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500

# Simulate data
X1 = np.random.normal(0, 1, n)
X2 = np.random.normal(0, 1, n)
T = (0.5 * X1 + 0.3 * X2 + np.random.normal(0, 0.5, n)) > 0
T = T.astype(int)
Y = 2 * T + 3 * X1 + 1 * X2 + np.random.normal(0, 1, n)

# Fit regression
X = sm.add_constant(np.column_stack([T, X1, X2]))
model = sm.OLS(Y, X).fit()

print(model.summary())

# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Residuals vs fitted
axes[0, 0].scatter(model.fittedvalues, model.resid, alpha=0.5, s=10)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Q-Q plot
from scipy import stats
stats.probplot(model.resid, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Scale-Location
axes[1, 0].scatter(model.fittedvalues, np.sqrt(np.abs(model.resid)), alpha=0.5, s=10)
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel('√|Residuals|')
axes[1, 0].set_title('Scale-Location')

# Residuals vs leverage
from statsmodels.graphics.regressionplots import plot_leverage_resid2
plot_leverage_resid2(model, ax=axes[1, 1])
axes[1, 1].set_title('Residuals vs Leverage')

plt.tight_layout()
plt.savefig('../figures/07-regression-diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7.7 Common Mistakes

1. **Assuming linearity**: Always check functional form
2. **Including post-treatment variables**: Don't control for mediators
3. **Ignoring heteroskedasticity**: Use robust standard errors
4. **Over-controlling**: Control for confounders, not colliders or mediators

---

## 7.8 Discussion Questions

1. **Education Returns**: Why might naive regression overestimate the return to education?

2. **Functional Form**: What happens if you include a quadratic term for age when the true relationship is linear?

3. **Measurement Error**: How does measurement error in a confounder affect your estimate?

4. **Multiple Regression**: When might adding more variables increase bias?

5. **Regression vs. Matching**: When might matching be preferable to regression?

---

## 7.9 Knowledge Check

### Multiple Choice

1. **Regression estimates the causal effect when:**
   - A) All confounders are controlled
   - B) The functional form is correct
   - C) Both A and B
   - D) Neither A nor B

2. **Omitted variable bias occurs when:**
   - A) We include too many variables
   - B) We exclude a confounder
   - C) We include a mediator
   - D) We include a collider

3. **Measurement error in a confounder causes:**
   - A) Upward bias
   - B) Downward bias (attenuation)
   - C) No bias
   - D) Inconsistency

4. **Including a post-treatment variable:**
   - A) Reduces bias
   - B) Increases bias
   - C) Has no effect
   - D) Depends on the situation

5. **The coefficient on treatment in multiple regression is:**
   - A) The correlation between T and Y
   - B) The partial effect of T on Y, holding X constant
   - C) The total effect of T on Y
   - D) The direct effect of T on Y

### Short Answer

6. **Explain why regression is sometimes called "adjustment for confounders."**

7. **What conditions must hold for regression to estimate causal effects?**

8. **How can you detect functional form misspecification?**

9. **Why might adding more variables to a regression increase bias?**

10. **Describe the difference between statistical and practical significance in regression.**

---

## 7.10 Summary

1. **Regression** can estimate causal effects under unconfoundedness
2. **Multiple regression** controls for observed confounders
3. **Omitted variable bias** is the main threat
4. **Functional form** matters for correct estimation
5. **Diagnostics** are essential for validating regression models

---

## 7.11 Further Reading

- Wooldridge, J.M. (2010). *Econometric Analysis of Cross Section and Panel Data*. MIT Press.
- Angrist, J.D. & Pischke, J.S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.